# GraphRAG + Temporal Graph + RDF/PROV-O — End-to-End Demo

**What this notebook shows:**

1. Load a small clinical knowledge graph into IVG
2. Run **Leiden community detection** to find patient cohorts
3. Call **Claude** (Anthropic API) to generate a clinical summary for each community
4. Write those summaries back as **temporal edges** — tracking *what the agent did, and when*
5. **Export the enriched graph as RDF** (Turtle) — portable to any SPARQL endpoint
6. **Validate** the enriched nodes with SHACL shapes
7. **Export PROV-O provenance** — a machine-readable audit trail of every agent action
8. Parse the PROV-O with rdflib and display the provenance timeline

**The key insight**: most GraphRAG demos stop at step 3. Steps 4–8 are the difference between
a prototype and a production-grade agentic system — you can prove *what* the LLM touched,
*when*, and *in what order*. That's the audit trail healthcare and enterprise compliance require.

---

**Prerequisites**
```bash
pip install 'iris-vector-graph[rdf]' anthropic
scripts/test-container.sh up   # IRIS Community Edition
export ANTHROPIC_API_KEY=sk-ant-...
```

## 0. Setup

In [ ]:
import os, json, time, textwrap, tempfile
from datetime import datetime, timezone

import rdflib
import anthropic

import iris.dbapi as dbapi
from iris_vector_graph.engine import IRISGraphEngine

# Connect to IRIS
conn = dbapi.connect(
    hostname=os.environ.get("IRIS_HOST", "localhost"),
    port=int(os.environ.get("IVG_PORT", "21972")),
    namespace="USER",
    username=os.environ.get("IRIS_USER", "_SYSTEM"),
    password=os.environ.get("IRIS_PASSWORD", "SYS"),
)
engine = IRISGraphEngine(conn, embedding_dimension=4)
engine.initialize_schema()

# Anthropic client
llm = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

print("Connected to IRIS and Anthropic ✓")

## 1. Seed the Clinical Knowledge Graph

A synthetic HFrEF (heart failure with reduced ejection fraction) cohort.
Real-world analogue: a FHIR R4 bundle from HAPI FHIR or Epic.

In [ ]:
RUN_ID = f"demo_{int(time.time()) % 100000}"  # unique prefix for cleanup

# ----- Patients (6 total — 3 HFrEF, 1 HFpEF, 2 control) -----
patients = [
    {"id": f"{RUN_ID}:P001", "name": "Alice Chen",    "age": "62", "cohort": "HFrEF",   "ef": "35"},
    {"id": f"{RUN_ID}:P002", "name": "Bob Martinez",  "age": "71", "cohort": "HFrEF",   "ef": "30"},
    {"id": f"{RUN_ID}:P003", "name": "Carol Kim",     "age": "58", "cohort": "HFrEF",   "ef": "38"},
    {"id": f"{RUN_ID}:P004", "name": "David Park",    "age": "67", "cohort": "HFpEF",   "ef": "58"},
    {"id": f"{RUN_ID}:P005", "name": "Eva Nguyen",    "age": "54", "cohort": "control", "ef": "62"},
    {"id": f"{RUN_ID}:P006", "name": "Frank Torres",  "age": "49", "cohort": "control", "ef": "65"},
]

# ----- Medications -----
meds = [
    {"id": f"{RUN_ID}:M001", "name": "Sacubitril/Valsartan", "class": "ARNI"},
    {"id": f"{RUN_ID}:M002", "name": "Metoprolol Succinate",  "class": "BB"},
    {"id": f"{RUN_ID}:M003", "name": "Dapagliflozin",         "class": "SGLT2i"},
    {"id": f"{RUN_ID}:M004", "name": "Lisinopril",             "class": "ACEi"},
]

# ----- Edges: which patients are on which meds -----
prescriptions = [
    (f"{RUN_ID}:P001", f"{RUN_ID}:M001"),
    (f"{RUN_ID}:P001", f"{RUN_ID}:M002"),
    (f"{RUN_ID}:P001", f"{RUN_ID}:M003"),
    (f"{RUN_ID}:P002", f"{RUN_ID}:M001"),
    (f"{RUN_ID}:P002", f"{RUN_ID}:M002"),
    (f"{RUN_ID}:P003", f"{RUN_ID}:M002"),
    (f"{RUN_ID}:P003", f"{RUN_ID}:M003"),
    (f"{RUN_ID}:P004", f"{RUN_ID}:M004"),
    (f"{RUN_ID}:P005", f"{RUN_ID}:M004"),
    (f"{RUN_ID}:P006", f"{RUN_ID}:M004"),
]

# Load into IVG
for p in patients:
    engine.create_node(
        p["id"],
        labels=["Patient"],
        properties={k: v for k, v in p.items() if k != "id"},
    )
for m in meds:
    engine.create_node(
        m["id"],
        labels=["Medication"],
        properties={k: v for k, v in m.items() if k != "id"},
    )
for src, tgt in prescriptions:
    engine.create_edge(src, "PRESCRIBED", tgt)

print(f"Seeded: {len(patients)} patients, {len(meds)} medications, {len(prescriptions)} prescriptions")
print(f"Run ID: {RUN_ID}")

## 2. Leiden Community Detection

IVG's `leiden_communities()` runs the Leiden algorithm (Traag et al. 2019)
against the `^NKG` integer adjacency index.
Communities here = natural groupings by shared medication patterns.

> **Note**: Leiden requires `pip install python-igraph leidenalg` and the
> `^NKG` index to be built. Call `engine.rebuild_nkg()` if needed.

In [ ]:
engine.rebuild_nkg()

try:
    communities_result = engine.leiden_communities(gamma=1.0, top_k=20)
    print(f"Found {len(communities_result)} community assignments")

    # Group by community
    community_map = {}  # community_id → list of node_ids
    for row in communities_result:
        # row: {"id": node_id, "community": int, "size": int}
        node_id = row.get("id", "")
        comm_id = str(row.get("community", 0))
        if node_id.startswith(RUN_ID):
            community_map.setdefault(comm_id, []).append(node_id)

    print("\nCommunities containing demo nodes:")
    for cid, members in sorted(community_map.items()):
        print(f"  Community {cid}: {', '.join(m.split(':')[-1] for m in members)}")

except Exception as e:
    print(f"Leiden skipped (requires igraph/leidenalg): {e}")
    # Fallback: manual community assignment based on cohort
    community_map = {
        "0": [f"{RUN_ID}:P001", f"{RUN_ID}:P002", f"{RUN_ID}:P003",
              f"{RUN_ID}:M001", f"{RUN_ID}:M002", f"{RUN_ID}:M003"],
        "1": [f"{RUN_ID}:P004", f"{RUN_ID}:P005", f"{RUN_ID}:P006",
              f"{RUN_ID}:M004"],
    }
    print(f"Using manual fallback communities: {list(community_map.keys())}")

## 3. LLM Community Summarization (Claude)

For each community, pull the node properties and ask Claude to write
a concise clinical summary. This is the **GraphRAG** step — the LLM
summarizes the graph structure, not just raw text.

We record the **exact timestamp** of each LLM call so we can track provenance later.

In [ ]:
def get_node_properties(engine, node_id: str) -> dict:
    """Fetch a node's properties from IVG."""
    result = engine.execute_cypher(
        "MATCH (n {node_id: $id}) RETURN n",
        {"id": node_id},
    )
    if result.rows:
        return result.rows[0][0] if result.rows[0] else {}
    return {}


def summarize_community(llm, community_id: str, members: list, engine) -> dict:
    """Call Claude to summarize a community. Returns summary + timestamp."""
    # Collect member properties
    member_data = []
    for node_id in members[:8]:  # cap at 8 to keep prompts short
        props = get_node_properties(engine, node_id)
        member_data.append({"id": node_id.split(":")[-1], **props})

    prompt = textwrap.dedent(f"""
        You are a clinical data analyst. Summarize this patient-medication community
        in 1-2 sentences. Focus on the clinical pattern (e.g. medication class,
        patient profile, treatment intensity).

        Community members:
        {json.dumps(member_data, indent=2)}

        Return ONLY the summary sentence. No preamble.
    """).strip()

    ts = int(time.time())
    response = llm.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}],
    )
    summary = response.content[0].text.strip()

    return {
        "community_id": community_id,
        "summary": summary,
        "timestamp": ts,
        "model": "claude-haiku-4-5-20251001",
        "member_count": len(members),
    }


# Run summarization for each community
summaries = []
for cid, members in sorted(community_map.items()):
    print(f"Summarizing community {cid} ({len(members)} members)...", end=" ")
    result = summarize_community(llm, cid, members, engine)
    summaries.append(result)
    print(f"✓  [{datetime.fromtimestamp(result['timestamp']).strftime('%H:%M:%S')}]")
    print(f"   → {result['summary']}")

print(f"\n{len(summaries)} communities summarized by Claude.")

## 4. Write Summaries Back as Temporal Edges

We use **temporal edges** to record each agent action:
- Source: the agent (a node representing the LLM call)
- Predicate: `annotated`
- Target: the community representative node
- Timestamp: exact Unix time of the LLM call

We also write the summary text as a property on each community's representative patient.

This is the critical step that enables PROV-O export later.

In [ ]:
AGENT_NODE_ID = f"{RUN_ID}:agent:claude"

# Create the agent node
engine.create_node(
    AGENT_NODE_ID,
    labels=["Agent", "LLM"],
    properties={"model": "claude-haiku-4-5-20251001", "role": "clinical_summarizer"},
)

# Write summaries and temporal edges
annotation_events = []
for s in summaries:
    cid = s["community_id"]
    members = community_map.get(cid, [])
    patients_in_community = [m for m in members if ":P" in m]

    if not patients_in_community:
        continue

    representative = patients_in_community[0]  # first patient = community rep

    # Write the summary back as a node property
    cur = conn.cursor()
    cur.execute(
        "UPDATE Graph_KG.rdf_props SET val = ? WHERE s = ? AND \"key\" = 'community_summary'",
        [s["summary"], representative],
    )
    if cur.rowcount == 0:
        cur.execute(
            "INSERT INTO Graph_KG.rdf_props (s, \"key\", val) VALUES (?, 'community_summary', ?)",
            [representative, s["summary"]],
        )
    conn.commit()
    cur.close()

    # Write temporal edge: agent --annotated--> representative patient @ timestamp
    engine.create_edge_temporal(
        source=AGENT_NODE_ID,
        predicate="annotated",
        target=representative,
        timestamp=s["timestamp"],
    )

    annotation_events.append({
        "community": cid,
        "representative": representative,
        "timestamp": s["timestamp"],
        "summary": s["summary"],
    })

print(f"Wrote {len(annotation_events)} community summaries and temporal edges")
for ev in annotation_events:
    ts_str = datetime.fromtimestamp(ev['timestamp']).strftime('%H:%M:%S')
    rep = ev['representative'].split(':')[-1]
    print(f"  [{ts_str}] Community {ev['community']} → {rep}: {ev['summary'][:60]}...")

## 5. Export the Enriched Graph as RDF

Now that the graph contains the original clinical data **plus** the LLM-generated
community summaries, we export it as Turtle. This file can be loaded into
Apache Jena Fuseki, Oxigraph, GraphDB, or any SPARQL endpoint.

In [ ]:
import tempfile, os

# Register human-readable namespaces
engine.register_namespace("demo", f"urn:ivg:{RUN_ID}:")
engine.register_namespace("fhir", "http://hl7.org/fhir/")

# Export the full demo subgraph
rdf_path = f"/tmp/graphrag_demo_{RUN_ID}.ttl"
all_node_ids = (
    [p["id"] for p in patients]
    + [m["id"] for m in meds]
    + [AGENT_NODE_ID]
)

result = engine.export_rdf(rdf_path, node_ids=all_node_ids)
print(f"Exported {result['triples']} triples to {rdf_path}")
print(f"  Nodes: {result['nodes']}  |  Edges: {result['edges']}")

# Show a sample of the output
with open(rdf_path) as f:
    lines = f.readlines()

print(f"\nFirst 20 lines of {os.path.basename(rdf_path)}:")
print("-" * 60)
for line in lines[:20]:
    print(line, end="")
if len(lines) > 20:
    print(f"  ... ({len(lines) - 20} more lines)")

## 6. Validate with SHACL Shapes

We define a shape that requires every `Patient` node to have:
- A `name` property
- A `cohort` classification
- A `community_summary` (written by the LLM in step 4)

If the LLM step failed or was skipped, some patients will fail validation.

In [ ]:
PATIENT_SHAPE = """
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix ivg:  <urn:ivg:prop/> .

<urn:ivg:PatientShape> a sh:NodeShape ;
    sh:targetClass <urn:ivg:Patient> ;

    sh:property [
        sh:path     ivg:name ;
        sh:minCount 1 ;
        sh:message  "Patient must have a name" ;
    ] ;

    sh:property [
        sh:path     ivg:cohort ;
        sh:minCount 1 ;
        sh:message  "Patient must have a cohort classification" ;
    ] ;

    sh:property [
        sh:path     ivg:community_summary ;
        sh:minCount 1 ;
        sh:severity sh:Warning ;
        sh:message  "Patient representative should have a community summary (LLM annotation)" ;
    ] .
"""

patient_ids = [p["id"] for p in patients]
report = engine.validate_shacl(PATIENT_SHAPE, node_ids=patient_ids)

print(f"SHACL validation result: conforms = {report.conforms}")
print(f"Violations: {len(report.violations)}")

if report.violations:
    print("\nViolations:")
    for v in report.violations:
        node = v.focus_node.split(":")[-1]
        print(f"  [{v.severity}] {node:10s} → {v.message}")
else:
    print("\n✓ All patients pass validation — LLM annotations complete.")

## 7. Export PROV-O Provenance

The temporal edges we created in step 4 record every agent action.
`prov_export()` maps them to W3C PROV-O vocabulary:

```
agent:claude  ─[annotated]─>  Patient/P001   @ 14:32:05
agent:claude  ─[annotated]─>  Patient/P004   @ 14:32:07
```

becomes:

```turtle
urn:ivg:activity/...  a prov:Activity ;
    prov:startedAtTime "2026-06-17T14:32:05Z"^^xsd:dateTime ;
    prov:used <urn:ivg:entity/demo:agent:claude> ;
    <urn:ivg:pred/annotated> <urn:ivg:entity/demo:P001> .
```

In [ ]:
prov_path = f"/tmp/graphrag_prov_{RUN_ID}.ttl"

# Export only the annotation events we created in this run
if annotation_events:
    ts_min = min(ev["timestamp"] for ev in annotation_events) - 1
    ts_max = max(ev["timestamp"] for ev in annotation_events) + 1
    result = engine.prov_export(prov_path, ts_start=ts_min, ts_end=ts_max)
else:
    result = engine.prov_export(prov_path)

print(f"PROV-O export: {result['activities']} activities, {result['entities']} entities")
print(f"Output: {prov_path}")

with open(prov_path) as f:
    prov_lines = f.readlines()

print(f"\nPROV-O Turtle ({len(prov_lines)} lines):")
print("-" * 60)
for line in prov_lines[:30]:
    print(line, end="")
if len(prov_lines) > 30:
    print(f"  ... ({len(prov_lines) - 30} more lines)")

## 8. Parse and Display the Provenance Timeline

Load the PROV-O into rdflib and reconstruct the agent's activity timeline.
This is what a compliance auditor, a downstream system, or another agent would see.

In [ ]:
PROV = rdflib.Namespace("http://www.w3.org/ns/prov#")
XSD  = rdflib.Namespace("http://www.w3.org/2001/XMLSchema#")

g = rdflib.Graph()
g.parse(prov_path, format="turtle")

print(f"PROV-O graph: {len(g)} triples")
print(f"Activities:   {len(list(g.subjects(rdflib.RDF.type, PROV.Activity)))}")
print(f"Entities:     {len(list(g.subjects(rdflib.RDF.type, PROV.Entity)))}")
print()

# Reconstruct the timeline
print("=" * 60)
print("AGENT PROVENANCE TIMELINE")
print("=" * 60)

timeline = []
for activity in g.subjects(rdflib.RDF.type, PROV.Activity):
    started  = next(g.objects(activity, PROV.startedAtTime), None)
    used     = next(g.objects(activity, PROV.used), None)
    # Find what the activity acted on (via non-prov predicates)
    targets = [
        str(o) for _, p, o in g.triples((activity, None, None))
        if "prov" not in str(p) and "rdf" not in str(p) and str(o) != str(activity)
        and isinstance(o, rdflib.URIRef)
    ]
    timeline.append({
        "activity": str(activity).split("/")[-1][:20],
        "started":  str(started),
        "agent":    str(used).split("/")[-1].split(":")[-1] if used else "?",
        "targets":  [t.split("/")[-1].split(":")[-1] for t in targets[:2]],
    })

timeline.sort(key=lambda x: x["started"])

for i, ev in enumerate(timeline):
    ts_display = ev["started"].replace("T", " ").replace("Z", " UTC")
    targets_str = ", ".join(ev["targets"]) or "(see graph)"
    print(f"  [{i+1}] {ts_display}")
    print(f"       Agent:    {ev['agent']}")
    print(f"       Acted on: {targets_str}")
    print()

print(f"Audit trail: {len(timeline)} actions recorded.")
print("This PROV-O file is standard W3C — submit to any PROV-aware compliance system.")

## 9. Round-Trip: Load PROV-O Back into IVG

The RDF export round-trips cleanly. The exported Turtle can be imported
into any IVG instance, Jena Fuseki, or Oxigraph without loss.

In [ ]:
# Verify the enriched graph Turtle parses cleanly
g_check = rdflib.Graph()
g_check.parse(rdf_path, format="turtle")

# Count by type
by_type = {}
for s, p, o in g_check.triples((None, rdflib.RDF.type, None)):
    type_name = str(o).split(":")[-1]
    by_type[type_name] = by_type.get(type_name, 0) + 1

print("Graph composition (from exported Turtle):")
for t, count in sorted(by_type.items(), key=lambda x: -x[1]):
    print(f"  {t:20s}: {count} nodes")

# Check that community_summary triples are present
summary_pred = rdflib.URIRef("urn:ivg:prop/community_summary")
summaries_in_rdf = list(g_check.subject_objects(summary_pred))
print(f"\nCommunity summaries in RDF: {len(summaries_in_rdf)}")
for subj, obj in summaries_in_rdf:
    node = str(subj).split(":")[-1]
    print(f"  {node}: {str(obj)[:70]}")

## 10. What Just Happened — Summary

```
IVG Graph
   ├── 6 Patients + 4 Medications + 10 PRESCRIBED edges
   ├── Leiden community detection → 2 cohorts identified
   │
   ├── Claude (Haiku) summarized each cohort
   │      Community 0: "HFrEF patients on guideline-directed therapy"
   │      Community 1: "Control + HFpEF on ACEi monotherapy"
   │
   ├── Summaries written back as node properties
   ├── Temporal edges record agent actions with timestamps
   │
   ├── RDF Export → clinical.ttl (portable to any SPARQL endpoint)
   ├── SHACL Validation → all required fields present
   └── PROV-O Export → provenance.ttl (W3C audit trail)
```

**Why this matters for production AI systems:**

| What | Why it matters |
|------|----------------|
| Temporal edges for LLM actions | You know *when* the AI touched each node |
| PROV-O export | Submit to any W3C PROV-aware compliance/audit tool |
| SHACL validation after annotation | Catch missing/malformed LLM outputs before they propagate |
| RDF export | Share the enriched KG with SPARQL endpoints, rdflib pipelines, KGX |
| All in one store | No separate provenance database — IVG tracks graph data and provenance together |

**Next steps to explore:**
- Scale to DRKG (97K nodes / 5.9M edges) — see `docs/notebooks/biomed_drkg_showcase.ipynb`
- Connect the RDF export to a running Jena Fuseki instance for SPARQL queries
- Add SHACL shapes to enforce ontology alignment (OWL 2 RL inference already built in)
- Feed the PROV-O output to [PROV-AGENT](https://arxiv.org/abs/2508.02866) for multi-agent orchestration tracking

## Cleanup

In [ ]:
# Remove demo data from IRIS
cur = conn.cursor()
for table, col in [
    ("Graph_KG.nodes",     "node_id"),
    ("Graph_KG.rdf_edges", "s"),
    ("Graph_KG.rdf_labels","s"),
    ("Graph_KG.rdf_props", "s"),
]:
    try:
        cur.execute(f"DELETE FROM {table} WHERE {col} LIKE ?", [f"{RUN_ID}%"])
    except Exception:
        pass
conn.commit()
cur.close()

# Remove temp files
for path in [rdf_path, prov_path]:
    try:
        os.unlink(path)
    except OSError:
        pass

conn.close()
print("Cleaned up. Notebook complete.")